In [4]:
import os
import time
import json
import math
import random
import argparse
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

from sklearn.metrics import confusion_matrix
import wandb


parser = argparse.ArgumentParser(description="Atri: fine-tune backbones with optional features")
parser.add_argument("--data_dir", type=str, default="/home/nagaraj/Garbage_classification_files/Garbage classification")
parser.add_argument("--output_dir", type=str, default="partB_results")
parser.add_argument("--backbones", nargs="+", default=["resnet50", "efficientnet_b0"])
parser.add_argument("--strategies", nargs="+", default=["last", "partial", "gradual", "full"])
parser.add_argument("--input_size", type=int, default=224)
parser.add_argument("--batch_size", type=int, default=16)
parser.add_argument("--epochs", type=int, default=40)
parser.add_argument("--lr", type=float, default=5e-5)
parser.add_argument("--weight_decay", type=float, default=1e-4)
parser.add_argument("--dropout", type=float, default=0.3)
parser.add_argument("--partial_freeze_k", type=int, default=4)
parser.add_argument("--gradual_unfreeze_every", type=int, default=8)
parser.add_argument("--patience", type=int, default=6)
parser.add_argument("--seed", type=int, default=42)
parser.add_argument("--use_wandb", action="store_true")
parser.add_argument("--project", type=str, default="Atri")
parser.add_argument("--entity", type=str, default="cs24s023-iitm-ac-in")
parser.add_argument("--only_run", type=str, default=None, help="optional: <backbone>_<strategy> to run single experiment")
parser.add_argument("--export_onnx", action="store_true", help="Export best model to ONNX")
parser.add_argument("--try_tensorrt", action="store_true", help="Attempt TensorRT export (requires tensorrt/torch2trt)")
parser.add_argument("--no_cuda", action="store_true", help="Force CPU")

args, unknown_args = parser.parse_known_args()
if unknown_args:
    print("Note: ignoring unknown args (likely from IPython/Jupyter):", unknown_args)


DATA_DIR = args.data_dir
OUTPUT_DIR = args.output_dir
BACKBONES = args.backbones
STRATEGIES = args.strategies
INPUT_SIZE = args.input_size
BATCH_SIZE = args.batch_size
EPOCHS = args.epochs
LR = args.lr
WEIGHT_DECAY = args.weight_decay
DROPOUT = args.dropout
PARTIAL_FREEZE_K = args.partial_freeze_k
GRADUAL_UNFREEZE_EVERY = args.gradual_unfreeze_every
PATIENCE = args.patience
SEED = args.seed
USE_WANDB = args.use_wandb
PROJECT = args.project
ENTITY = args.entity

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

if USE_WANDB:
    try:
        wandb.login()
    except Exception as e:
        print("W&B login failed:", e)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if (torch.cuda.is_available() and not args.no_cuda) else "cpu")
print("Device:", device)

def save_json(obj, p):
    Path(p).parent.mkdir(parents=True, exist_ok=True)
    with open(p, "w") as f:
        json.dump(obj, f, indent=2)


def make_dataloaders(data_dir, input_size=INPUT_SIZE, batch_size=BATCH_SIZE, seed=SEED, augment=True):
    train_dir = os.path.join(data_dir, "train")
    test_dir = os.path.join(data_dir, "test")
    assert os.path.isdir(train_dir), f"Train dir not found: {train_dir}"
    assert os.path.isdir(test_dir), f"Test dir not found: {test_dir}"

    normalize = transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])

    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(input_size, scale=(0.6,1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(20),
        transforms.ColorJitter(0.4,0.4,0.4,0.1),
        transforms.ToTensor(),
        normalize
    ])

    val_tf = transforms.Compose([
        transforms.Resize((input_size,input_size)), transforms.ToTensor(), normalize
    ])

    train_ds = datasets.ImageFolder(train_dir, transform=train_tf)
    test_ds  = datasets.ImageFolder(test_dir, transform=val_tf)

    targets = np.array([s[1] for s in train_ds.samples])
    classes = train_ds.classes
    rng = np.random.RandomState(seed)
    train_idx, val_idx = [], []
    for c in np.unique(targets):
        idxs = np.where(targets == c)[0]
        rng.shuffle(idxs)
        n_train = max(1, int(0.8 * len(idxs)))
        train_idx += idxs[:n_train].tolist()
        val_idx += idxs[n_train:].tolist()

    train_loader = DataLoader(Subset(train_ds, train_idx), batch_size=batch_size, shuffle=True, pin_memory=True)
    val_loader = DataLoader(Subset(train_ds, val_idx), batch_size=batch_size, shuffle=False, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=True)
    return train_loader, val_loader, test_loader, classes


def build_model(backbone, num_classes, dropout=DROPOUT):
    backbone = backbone.lower()
    if backbone == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_features, num_classes))
    elif backbone == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_features, num_classes))
    else:
        raise ValueError("Unsupported backbone: " + backbone)
    return model

def apply_strategy(model, strategy, partial_freeze_k=PARTIAL_FREEZE_K):
    if strategy == "last":
        for p in model.parameters():
            p.requires_grad = False
        if hasattr(model, "fc"):
            for p in model.fc.parameters():
                p.requires_grad = True
        elif hasattr(model, "classifier"):
            for p in model.classifier.parameters():
                p.requires_grad = True

    elif strategy == "partial":
        children = list(model.children())
        for i, child in enumerate(children):
            freeze = (i < partial_freeze_k)
            for p in child.parameters():
                p.requires_grad = not freeze
      
        if hasattr(model, "fc"):
            for p in model.fc.parameters():
                p.requires_grad = True
        elif hasattr(model, "classifier"):
            for p in model.classifier.parameters():
                p.requires_grad = True

    elif strategy == "gradual":
        for p in model.parameters():
            p.requires_grad = False
        if hasattr(model, "layer4"):
            for p in model.layer4.parameters():
                p.requires_grad = True
        if hasattr(model, "fc"):
            for p in model.fc.parameters():
                p.requires_grad = True
        elif hasattr(model, "classifier"):
            for p in model.classifier.parameters():
                p.requires_grad = True

    elif strategy == "full":
        for p in model.parameters():
            p.requires_grad = True
    else:
        raise ValueError("Unknown strategy: " + strategy)
    return model


def train_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for imgs, labels in loader:
        imgs = imgs.to(device); labels = labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(scaler is not None)):
            outs = model(imgs)
            loss = criterion(outs, labels)
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        preds = outs.argmax(1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
    return running_loss / max(1, total), correct / max(1, total)

@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    for imgs, labels in loader:
        imgs = imgs.to(device); labels = labels.to(device)
        outs = model(imgs)
        loss = criterion(outs, labels)
        running_loss += loss.item() * imgs.size(0)
        preds = outs.argmax(1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())
    if total == 0:
        return 0.0, 0.0, np.array([]), np.array([])
    return running_loss / total, correct / total, torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()


def run_experiment(backbone, strategy):
    train_loader, val_loader, test_loader, classes = make_dataloaders(DATA_DIR)
    num_classes = len(classes)

    model = build_model(backbone, num_classes)
    model = apply_strategy(model, strategy)
    model = model.to(device)

    print(f"Model {backbone} — trainable params:",
          sum(p.numel() for p in model.parameters() if p.requires_grad),
          "/", sum(p.numel() for p in model.parameters()))

    criterion = nn.CrossEntropyLoss()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

    scaler = torch.cuda.amp.GradScaler() if device.type == "cuda" else None

    best_val = -1
    for epoch in range(1, EPOCHS+1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, device, scaler)
        val_loss, val_acc, _, _ = eval_epoch(model, val_loader, criterion, device)
        scheduler.step(val_acc)
        print(f"[{backbone}/{strategy}] Epoch {epoch}: train_acc={tr_acc:.3f}, val_acc={val_acc:.3f}")

        if val_acc > best_val:
            best_val = val_acc
            torch.save(model.state_dict(), f"{OUTPUT_DIR}/{backbone}_{strategy}_best.pth")

    print("Best val acc:", best_val)

def main():
    pairs = [(bb, strat) for bb in BACKBONES for strat in STRATEGIES]
    if args.only_run:
        pairs = [tuple(args.only_run.split("_", 1))]
    for bb, strat in pairs:
        print("\n" + "="*80)
        print(f"Running: backbone={bb} | strategy={strat}")
        try:
            run_experiment(bb, strat)
        except Exception as e:
            print(f"Experiment {bb}_{strat} failed:", e)

if __name__ == "__main__":
    main()


Note: ignoring unknown args (likely from IPython/Jupyter): ['-f', '/home/nagaraj/.local/share/jupyter/runtime/kernel-6a11ac57-6472-40c1-ae19-dc352696b3f6.json']
Device: cuda

Running: backbone=resnet50 | strategy=last
Model resnet50 — trainable params: 12294 / 23520326


/tmp/ipykernel_1880565/2100966757.py:269: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if device.type == "cuda" else None
/tmp/ipykernel_1880565/2100966757.py:212: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(scaler is not None)):


[resnet50/last] Epoch 1: train_acc=0.242, val_acc=0.351
[resnet50/last] Epoch 2: train_acc=0.363, val_acc=0.454
[resnet50/last] Epoch 3: train_acc=0.445, val_acc=0.568
[resnet50/last] Epoch 4: train_acc=0.511, val_acc=0.667
[resnet50/last] Epoch 5: train_acc=0.551, val_acc=0.672
[resnet50/last] Epoch 6: train_acc=0.572, val_acc=0.684
[resnet50/last] Epoch 7: train_acc=0.586, val_acc=0.726
[resnet50/last] Epoch 8: train_acc=0.615, val_acc=0.719
[resnet50/last] Epoch 9: train_acc=0.613, val_acc=0.714
[resnet50/last] Epoch 10: train_acc=0.629, val_acc=0.701
[resnet50/last] Epoch 11: train_acc=0.645, val_acc=0.728
[resnet50/last] Epoch 12: train_acc=0.662, val_acc=0.711
[resnet50/last] Epoch 13: train_acc=0.658, val_acc=0.748
[resnet50/last] Epoch 14: train_acc=0.656, val_acc=0.736
[resnet50/last] Epoch 15: train_acc=0.675, val_acc=0.758
[resnet50/last] Epoch 16: train_acc=0.673, val_acc=0.753
[resnet50/last] Epoch 17: train_acc=0.698, val_acc=0.760
[resnet50/last] Epoch 18: train_acc=0.68